# The Kessler Game

**Kessler** is an asteroids game built for testing artificial intelligence. Ships fly around a map full of drifting asteroids, shooting them to score. A hit splits an asteroid into smaller pieces, and crashing into one costs a life. It was made by the engineering company Thales and is the game used in the **Explainable Fuzzy Competition**, where teams write AI pilots and race them against each other.

The twist is that you don't usually fly the ship yourself. You write a **controller**: a small piece of Python that looks at the game 30 times a second and decides what the ship should do. In this notebook you'll fly with the keyboard and a game controller first, then write controllers of your own and test them with and without graphics.

The library is imported as `kesslergame`.

# Scenarios

Every match starts from a **`Scenario`**: the size of the map, where the ships start, how many asteroids there are and how long the match lasts.

- `num_asteroids` places that many asteroids at random. Passing a `seed` makes the "random" layout the same every time, which is useful when comparing controllers.
- `ship_states` is a list with one dictionary per ship: its starting `position`, its `angle` (the way it points), its `lives`, its `team`, and optionally `mines_remaining`. Ships carry no mines unless you give them some.
- Positions are `(x, y)` with **y pointing up**, like a graph in maths class. Angles are in degrees: 0 points right and 90 points up.

In [ ]:
from kesslergame import KesslerGame, Scenario, KeyboardController

practice = Scenario(
  name="Practice",
  num_asteroids=6,
  ship_states=[{"position": (500, 400), "angle": 90, "lives": 3, "team": 1, "mines_remaining": 3}],
  map_size=(1000, 800),
  time_limit=60,
  seed=1,
)

print("Map size:", practice.map_size)
print("Asteroids:", len(practice.asteroids()))
print("Time limit:", practice.time_limit, "seconds")

# Flying with the Keyboard

`KesslerGame().run()` plays a scenario on screen, with one controller for each ship. The **`KeyboardController`** lets you be the controller:

| Key | Action |
|---|---|
| Up or W | Thrust forwards |
| Down or S | Reverse |
| Left or A, Right or D | Turn |
| Space | Fire |
| M | Drop a mine |

Run the cell and fly! The match ends after 60 seconds, when you run out of lives, or when every asteroid is gone. Mines explode three seconds after you drop them, so get clear! When the match ends, your score appears on the console tab (>_) under the game.

In [ ]:
game = KesslerGame()
score, perf = game.run(scenario=practice, controllers=[KeyboardController()])

team = score.teams[0]
print(f"{team.asteroids_hit} asteroids hit, {team.accuracy:.0%} accuracy, {team.deaths} deaths")

# Game Controllers

If you have an Xbox, PlayStation or Switch Pro controller, plug it in or pair it, and press any button so the browser can see it. Then use the **`GamepadController`**:

| Control | Action |
|---|---|
| Left stick up and down | Thrust and reverse |
| Right stick left and right | Turn |
| Right trigger | Fire |
| Right bumper | Drop a mine |

If no controller is found, the ship just waits, so you can press a button after the match starts. With two controllers, `GamepadController(index=1)` uses the second one. The [[8962E963-B160-403F-AB64-553FE1C2FF8D|Game Controllers]] notebook has more about using them in Python.

In [ ]:
from kesslergame import GamepadController

score, perf = KesslerGame().run(scenario=practice, controllers=[GamepadController()])

team = score.teams[0]
print(f"{team.asteroids_hit} asteroids hit, {team.accuracy:.0%} accuracy, {team.deaths} deaths")

# Writing a Controller

A controller is a class that inherits from **`KesslerController`**. It needs a `name`, and an **`actions()`** method that the game calls 30 times a second. `actions()` returns four values:

1. **thrust**: from -480 (full reverse) to 480 (full speed ahead)
2. **turn rate**: from -180 to 180 degrees per second, where positive turns **anticlockwise** (left)
3. **fire**: `True` to shoot
4. **drop mine**: `True` to drop a mine

This first controller doesn't look at the game at all. It spins on the spot and fires constantly. It's not clever, but it's a surprisingly hard opponent for asteroids that drift too close.

In [ ]:
from kesslergame import KesslerController

class Spinner(KesslerController):
  name = "Spinner"

  def actions(self, ship_state, game_state):
    thrust = 0.0
    turn_rate = 90.0
    fire = True
    drop_mine = False
    return thrust, turn_rate, fire, drop_mine

short_match = Scenario(num_asteroids=6, ship_states=[{"position": (500, 400)}], time_limit=15, seed=1)
score, perf = KesslerGame().run(scenario=short_match, controllers=[Spinner()])

team = score.teams[0]
print(f"{team.asteroids_hit} asteroids hit, {team.accuracy:.0%} accuracy, {team.deaths} deaths")

# Reading the Game

A clever controller looks before it acts. `actions()` receives two arguments:

- **`ship_state`** describes your own ship: `position`, `heading` (the way it points), `velocity`, `lives_remaining`, `can_fire` and more.
- **`game_state`** describes everything else: lists of `asteroids`, `ships`, `bullets` and `mines`, each with a `position`, plus the `time` and the `map_size`.

Adding the types `ShipState` and `GameState` to the arguments, as below, lets the editor suggest every field as you type `ship_state.`.

This controller finds the nearest asteroid, works out the angle to it with `math.atan2`, and turns towards it. The **aim error** is how far the ship has to turn, from -180 to 180 degrees. It only fires once the error is small.

In [ ]:
import math
from kesslergame.state_models import ShipState, GameState

class Aimer(KesslerController):
  name = "Aimer"

  def __init__(self, turn_speed=6, aim_tolerance=10):
    self.turn_speed = turn_speed
    self.aim_tolerance = aim_tolerance

  def actions(self, ship_state: ShipState, game_state: GameState):
    x, y = ship_state.position
    nearest = min(game_state.asteroids, key=lambda a: math.dist(a.position, (x, y)))
    target = math.degrees(math.atan2(nearest.y - y, nearest.x - x))
    error = (target - ship_state.heading + 180) % 360 - 180
    turn_rate = max(-180, min(180, error * self.turn_speed))
    return 0.0, turn_rate, abs(error) < self.aim_tolerance, False

score, perf = KesslerGame().run(scenario=short_match, controllers=[Aimer()])

team = score.teams[0]
print(f"{team.asteroids_hit} asteroids hit, {team.accuracy:.0%} accuracy, {team.deaths} deaths")

# Running Without Graphics

Watching a match is fun, but testing a controller properly means running many matches, and waiting 30 seconds for each is slow. **`TrainerEnvironment`** runs the same game with no graphics, as fast as the computer can go. A 30-second match takes a fraction of a second.

It returns the same **score** as `KesslerGame`. `score.teams` has one entry per team, with `asteroids_hit`, `accuracy` (the fraction of shots that hit) and `deaths`.

Drag the sliders to tune the Aimer. `TURN_SPEED` is how sharply it turns towards its target, and `AIM_TOLERANCE` is how close to on-target it must be before it fires. Each change runs five matches with different seeds and averages the results. Can you find settings that hit more and die less?

In [ ]:
from kesslergame import TrainerEnvironment

TURN_SPEED = 6 #@param {type:"slider", min:1, max:20, step:1}
AIM_TOLERANCE = 10 #@param {type:"slider", min:2, max:45, step:1}

hits, accuracy, deaths = [], [], []
for seed in range(5):
  test = Scenario(num_asteroids=8, ship_states=[{"position": (500, 400)}], time_limit=30, seed=seed)
  score, perf = TrainerEnvironment().run(scenario=test, controllers=[Aimer(TURN_SPEED, AIM_TOLERANCE)])
  team = score.teams[0]
  hits.append(team.asteroids_hit)
  accuracy.append(team.accuracy)
  deaths.append(team.deaths)

print(f"Average asteroids hit: {sum(hits) / 5:.1f}")
print(f"Average accuracy: {sum(accuracy) / 5:.0%}")
print(f"Average deaths: {sum(deaths) / 5:.1f}")

# Head to Head

A scenario can hold several ships, each with its own controller and team. Ships can't shoot each other, but they can crash into each other or get caught in each other's mine blasts.

Here you fly the green ship (team 1) against the Aimer (team 2), using the same keys as before. Both ships start with three mines. Who hits more asteroids in 60 seconds? The console tab shows both teams' results after the match.

In [ ]:
versus = Scenario(
  num_asteroids=10,
  ship_states=[
    {"position": (300, 400), "angle": 90, "lives": 3, "team": 1, "mines_remaining": 3},
    {"position": (700, 400), "angle": 90, "lives": 3, "team": 2, "mines_remaining": 3},
  ],
  time_limit=60,
)

score, perf = KesslerGame().run(scenario=versus, controllers=[KeyboardController(), Aimer()])

for team, who in zip(score.teams, ["You", "Aimer"]):
  print(f"{who}: {team.asteroids_hit} hits, {team.accuracy:.0%} accuracy, {team.deaths} deaths")

# Check Your Understanding